# Agent-to-Agent Protocols (A2A)

Companion notebook for the [A2A lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/13-agent-to-agent-protocols).

**The idea in one sentence.** A2A lets **independent agents** — different frameworks, vendors, memories — **discover** each other (via an Agent Card) and **delegate** stateful, long-running **tasks** as peers; it is the *horizontal* complement to MCP's *vertical* tool access.

We build a minimal in-process A2A: an Agent Card, a Task state machine, typed message Parts, and two toy agents that run a delegation end to end.

> **To save your work:** click **Copy to Drive** at the top, or File -> Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)
import json

## 1. The Agent Card (discovery), from scratch

A remote agent publishes a small JSON document (by convention at `/.well-known/agent.json`) advertising its skills and endpoint. A client reads it *before* delegating — it never sees the remote agent's internal tools or prompts.

In [ ]:
flight_agent_card = {
    'name': 'Flight Booking Agent',
    'description': 'Searches and books flights',
    'url': 'https://air.example.com/a2a',
    'skills': [{'id': 'book_flight', 'description': 'Book a flight given route and dates'}],
    'authentication': {'schemes': ['bearer']},
}
print(json.dumps(flight_agent_card, indent=2))

## 2. The Task lifecycle

A2A work is a **Task** with an explicit state machine. Long-running tasks can pause to ask the client a question (`input-required`) before resuming. We encode the legal transitions and a helper that enforces them.

In [ ]:
TRANSITIONS = {
    'submitted':      {'working'},
    'working':        {'input-required', 'completed', 'failed'},
    'input-required': {'working', 'canceled'},
    'completed':      set(),
    'failed':         set(),
    'canceled':       set(),
}

class Task:
    def __init__(self, text):
        self.state = 'submitted'; self.history = ['submitted']
        self.messages = [{'role': 'user', 'parts': [{'type': 'text', 'text': text}]}]
        self.artifacts = []
    def to(self, new):
        assert new in TRANSITIONS[self.state], 'illegal ' + self.state + ' -> ' + new
        self.state = new; self.history.append(new)

t = Task('Book LAX -> JFK on Mar 3')
print('start:', t.state)

## 3. Two agents, one delegation

A **client/orchestrator** delegates to the **flight agent**. The remote agent runs its own loop (it might call tools over MCP), pauses once for a clarification, then returns an **artifact**. Messages are lists of typed **parts** (`text`, `data`, `file`).

In [ ]:
def text_part(s):  return {'type': 'text', 'text': s}
def data_part(d):  return {'type': 'data', 'data': d}

def flight_agent(task, answer=None):
    # remote agent's turn; returns ('input-required', question) or ('completed', artifact)
    if task.state == 'submitted':
        task.to('working')
        return 'input-required', text_part('Aisle or window seat?')
    if task.state == 'input-required':
        task.to('working')
        seat = answer or 'aisle'
        return 'completed', data_part({'confirmation': 'AB123', 'route': 'LAX-JFK', 'seat': seat})

def orchestrate(card, request, seat_pref):
    print('discovered:', card['name'], '-> skill', card['skills'][0]['id'])
    task = Task(request)
    status, part = flight_agent(task)          # 1st turn -> asks a question
    task.to('input-required')
    print('remote asks:', part['text'])
    status, part = flight_agent(task, answer=seat_pref)  # answer -> completes
    task.to(status)
    task.artifacts.append(part)
    return task

task = orchestrate(flight_agent_card, 'Book LAX -> JFK on Mar 3', seat_pref='window')
print('final state:', task.state)
print('artifact   :', task.artifacts[-1]['data'])
print('history    :', ' -> '.join(task.history))

**What to notice.** The client only ever spoke the *protocol* — a card, a task, messages, an artifact. It never touched the flight agent's internal tools or reasoning. That opacity is the point: either side can be any framework or vendor.

### Visualize the lifecycle

In [ ]:
states = task.history
plt.figure(figsize=(8, 2.4))
plt.plot(range(len(states)), [0]*len(states), '-o', color='#6366f1')
for i, s in enumerate(states):
    plt.annotate(s, (i, 0), textcoords='offset points', xytext=(0, 10),
                 ha='center', color='#e2e8f0')
plt.yticks([]); plt.ylim(-0.5, 0.6); plt.xlim(-0.5, len(states)-0.5)
plt.title('A2A task state over the delegation')
plt.show()

**What to notice.** The task passed through `input-required` and back to `working` before `completed`. Treating A2A as a synchronous request/response would break exactly at that pause — long-running, multi-turn tasks are first-class in the protocol.

## 4. The library way, and MCP vs A2A

Real A2A rides **HTTP + JSON-RPC** with **SSE** for streaming; SDKs (e.g. the `a2a` project) handle cards, tasks, and transport. The two protocols compose:

| | MCP | A2A |
|---|---|---|
| Connects agent to | tools & data (vertical) | other agents (horizontal) |
| Other side is | a passive resource | an autonomous peer |
| Unit | tool call | stateful task |
| Transport | JSON-RPC | HTTP + JSON-RPC + SSE |

An agent is usually **both** — an MCP *client* to its tools and an A2A *peer* to other agents.

## 5. Your turn

Add a **failure path**: if the requested route is unavailable, the flight agent should move the task to `failed` with an explanatory text part. Implement `flight_agent_v2(task, available)` and verify the state machine rejects any transition out of `failed`.

In [ ]:
def flight_agent_v2(task, available=True, answer=None):
    # TODO(you): if not available, go working -> failed with a reason part
    raise NotImplementedError

# t = Task('Book JFK -> nowhere'); t.to('working')
# status, part = flight_agent_v2(t, available=False)
# assert t.state == 'failed'
print('implement the failure path, then check against the solution')

<details><summary>Solution</summary>

```python
def flight_agent_v2(task, available=True, answer=None):
    if not available:
        task.to('failed')
        return 'failed', text_part('No route available for that request')
    return flight_agent(task, answer)

t = Task('Book JFK -> nowhere'); t.to('working')
status, part = flight_agent_v2(t, available=False)
assert t.state == 'failed'
try:
    t.to('working')          # illegal: failed is terminal
except AssertionError as e:
    print('correctly rejected:', e)
```

Terminal states (`failed`, `completed`, `canceled`) have no outgoing transitions — the state machine enforces it.
</details>

## 6. Key takeaways

- **A2A** lets independent agents **discover** (Agent Card) and **delegate** stateful **tasks** to each other as opaque peers.
- Tasks have a real **lifecycle** (`submitted -> working -> input-required -> completed/failed`) — they are long-running and multi-turn.
- A2A is the **horizontal** complement to **MCP's vertical** tool access; agents are commonly both.
- The protocol is the easy part — **trust, authorization, and failure handling** are the real work.

Next: [Deploying Agents](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/11-deploying-agents) to run these systems in production.